# Evaluación 101
### La brújula antes de modelar: split temporal, WAPE, MASE y BIAS

`Fase 3 · Video 11` — serie **Forecasting de Ventas de la A a la Z**

Antes de tocar un solo modelo necesitamos una **brújula**: cómo partir los datos sin hacer trampa y
con qué números decidir si un pronóstico es bueno. Todo el zoológico de modelos que viene (Baselines,
ARIMA, Prophet, GBM, Foundation…) se juzga con lo que fijamos aquí. Sin esto, comparar modelos es
opinión, no evidencia.

---
## 1. ¿Por qué una brújula *antes* de modelar?

Un error clásico de las series de tutoriales es mostrar modelos y *después* explicar cómo se miden.
El problema: cuando dices *"Prophet gana con WAPE 0.18"* pero el espectador aún no sabe qué es WAPE,
la comparación no significa nada.

Aquí invertimos el orden. Fijamos **tres decisiones** que no volveremos a discutir:

1. **Cómo partimos los datos** — un split que respeta el tiempo (nunca entrenar con el futuro).
2. **Con qué métricas medimos** — WAPE, MASE y BIAS, cada una responde una pregunta distinta.
3. **Cómo las leemos** — qué valor es "bueno" y qué nos dice cada una del negocio.

> ⚠️ El split de este video es **simple** (un solo corte train/test). Es honesto pero optimista:
> un solo corte puede engañar. En el **Video 20 (Validación Cruzada Temporal)** lo convertimos en una
> **distribución** de error con walk-forward. Empezamos simple para no cargar todo de golpe.

---
## 2. El split que respeta el tiempo

En datos tabulares mezclas filas al azar para el train/test. **En series de tiempo eso es hacer trampa:**
usarías el futuro para predecir el pasado. La regla es sagrada — **el test siempre va DESPUÉS del train**.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reusar las métricas del framework (fuente única de verdad: src/metrics.py)
ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / 'src').exists())
sys.path.insert(0, str(ROOT))
from src.metrics import wape, mase, bias

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0', 'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, ORANGE, GREEN, RED = '#2563EB', '#EA580C', '#16A34A', '#DC2626'


def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")


df = pd.read_csv(find_csv(), parse_dates=['date']).sort_values('date')

# Serie semanal del total (quitamos semanas parciales de los bordes)
y = df.groupby(pd.Grouper(key='date', freq='W-MON'))['units_sold'].sum().iloc[1:-1]

# Split temporal: las últimas H semanas son el test
H = 13
train, test = y.iloc[:-H], y.iloc[-H:]
print(f"Total: {len(y)} semanas  |  train: {len(train)}  |  test (futuro): {len(test)}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(train.index, train.values, color=BLUE, label='train (pasado)')
ax.plot(test.index, test.values, color=RED, linewidth=2.2, label='test (futuro)')
ax.axvline(train.index[-1], color='#64748B', linestyle='--', linewidth=1)
ax.set_title('Split temporal — el test SIEMPRE va después del train')
ax.set_ylabel('unidades / semana')
ax.legend(loc='upper left')
fig.tight_layout()

---
## 3. Las tres métricas núcleo

Cada métrica responde una pregunta distinta. No hay una sola "buena": se leen juntas.

| Métrica | Fórmula | Pregunta que responde | Lectura |
|---|---|---|---|
| **WAPE** | $\dfrac{\sum\lvert real-pred\rvert}{\sum\lvert real\rvert}$ | ¿Qué tan grande es mi error vs. el volumen? | Fracción; **robusta a ceros** (a diferencia del MAPE) |
| **MASE** | $\dfrac{\text{MAE modelo}}{\text{MAE naive estacional}}$ | ¿Le gano a lo trivial? | **< 1** vence al naive; **> 1** no lo supera |
| **BIAS** | $\dfrac{\sum(pred-real)}{\sum\lvert real\rvert}$ | ¿Me quedo corto o me paso? | **+** sobre-pronostica (exceso de stock); **−** se queda corto (quiebres) |

WAPE mide *magnitud*, MASE mide *si vale la pena el modelo*, y BIAS mide *dirección* del error —
crítico en inventario: no cuesta lo mismo pasarse que quedarse corto.

In [ ]:
# Dos baselines simples para ver las métricas en acción (el detalle es del Video 12)
m = 52  # estacionalidad semanal ~ 1 año
naive = np.repeat(train.iloc[-1], H)                               # repite el último valor
snaive = (np.resize(train.iloc[-m:].values, H) if len(train) >= m  # repite el patrón del último año
          else np.repeat(train.mean(), H))

rows = []
for name, fc in [('Naive', naive), ('Seasonal Naive', snaive)]:
    rows.append({
        'modelo': name,
        'WAPE': wape(test.values, fc),
        'MASE': mase(test.values, fc, train.values, min(m, len(train) - 1)),
        'BIAS': bias(test.values, fc),
    })

tabla = pd.DataFrame(rows).set_index('modelo')
tabla.style.format({'WAPE': '{:.1%}', 'MASE': '{:.2f}', 'BIAS': '{:+.1%}'})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
colors = [RED if v >= 1 else GREEN for v in tabla['MASE']]
ax.barh(tabla.index, tabla['MASE'], color=colors)
ax.axvline(1.0, color='#334155', linestyle='--', linewidth=1.5)
ax.text(1.02, -0.4, 'MASE = 1 (empata al naive)', color='#334155', fontsize=9)
ax.set_xlabel('MASE')
ax.set_title('¿Le gana al naive estacional? (MASE < 1 = sí)')
fig.tight_layout()

---
## 4. Cómo leer estos números

- **WAPE** te da la magnitud del error en términos de negocio ("me equivoco en ~X% del volumen").
- **MASE** es el juez más honesto: si tu modelo sofisticado tiene MASE ≥ 1, **no vale la pena** —
  un naive estacional lo iguala gratis. Es el listón que fijamos en el Video 12.
- **BIAS** revela el riesgo operativo: un modelo con buen WAPE pero BIAS muy positivo está inflando
  el inventario de forma sistemática.

Ninguna se lee sola. Un pronóstico con WAPE bajo y BIAS alto sigue siendo peligroso.

---
## 5. Puente al Video 12 — Baselines Inteligentes

Ya tenemos la **brújula**: un split honesto y tres métricas que interpretamos. En el **Video 12**
la usamos de inmediato para fijar el **listón** (Seasonal Naive, Holt-Winters) que todo modelo
posterior deberá superar (MASE < 1).

Y recuerda: este split único se queda corto para decisiones serias. En el **Video 20** lo
robustecemos con **validación cruzada temporal (walk-forward)**, pasando de *un* número de error
a una *distribución*.

### Próximo video
**Video 12 — Baselines Inteligentes: el listón que todo modelo debe superar**